# Fase 2: exploración y calidad de series

Este notebook deja constancia de cómo se pasa del **panel por centro** de la fase 1 al
**panel de series** (`series_id`), con los cambios de código de centro enlazados, y de cómo se mide
la calidad de cada serie para decidir cuáles se podrán prever.

Todas las celdas llaman a funciones del paquete `prevision` y no tienen lógica propia, así que el
notebook y el pipeline (`python -m prevision.pipeline`) no pueden divergir. La última celda lo
comprueba.

**Qué no hace esta fase:** no corrige ni suaviza saltos, no imputa cursos y no entrena ningún modelo.
Los saltos solo se marcan.

In [1]:
import tempfile
from pathlib import Path

import pandas as pd
from IPython.display import display

from prevision.config import JUMP_RATIO, LINKS_FILE, MIN_SEASONS, RAW_FILE
from prevision.linking import apply_links, detect_candidates, load_links
from prevision.pipeline import build_series, run_pipeline
from prevision.quality import series_quality

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)

# Los ficheros del notebook van a una carpeta temporal: no tocan data/processed/.
out_dir = Path(tempfile.mkdtemp())
panel = run_pipeline(RAW_FILE, out_dir)
print(f"Panel de la fase 1: {panel.shape[0]} filas, {panel[['university', 'center_code']].drop_duplicates().shape[0]} centros")
print(f"Parámetros: MIN_SEASONS={MIN_SEASONS}, JUMP_RATIO={JUMP_RATIO}")

Panel de la fase 1: 1584 filas, 144 centros
Parámetros: MIN_SEASONS=4, JUMP_RATIO=1.5


## 1. Qué hay en el panel

El panel es **rectangular**: todos los centros tienen una fila por curso y un curso sin dato es una
fila con `enrolled = NaN` (no hay ceros ni filas ausentes). Un centro que abre tarde o cierra antes
se ve como NaN al principio o al final. Cualquier cálculo de "primer curso" o "último curso" debe
usar solo los cursos con matrícula informada.

In [2]:
panel.groupby("year_start")["enrolled"].agg(["size", "count", "sum"]).rename(
    columns={"size": "centros", "count": "con_dato", "sum": "matriculados"}
)

,centros,con_dato,matriculados
year_start,,,
2015,144,112,45715.0
2016,144,112,45553.0
2017,144,112,45919.0
2018,144,112,47004.0
2019,144,113,47994.0
2020,144,112,49175.0
2021,144,112,50432.0
2022,144,114,52048.0
2023,144,117,53629.0


Calidad de las series **sin enlazar** (cada centro es su propia serie). Es el punto de partida:
cuántos centros tienen historia suficiente antes de unir nada.

In [3]:
no_links = load_links(out_dir / "sin_enlaces.csv")  # fichero inexistente: tabla vacía
quality_unlinked = series_quality(apply_links(panel, no_links))
print("Series sin enlazar:", len(quality_unlinked))
print("Activas en el último curso:", int(quality_unlinked["active_last"].sum()))
print("Elegibles:", int(quality_unlinked["eligible"].sum()))
display(quality_unlinked["n_seasons"].value_counts().sort_index().rename("series").to_frame().T)
display(quality_unlinked.loc[quality_unlinked["active_last"], "size_last"].describe().round(0).to_frame().T)

Series sin enlazar: 144
Activas en el último curso: 124
Elegibles: 109


n_seasons,1,2,3,4,5,6,7,8,9,10,11
series,12,6,6,7,1,2,5,2,4,4,95


,count,mean,std,min,25%,50%,75%,max
size_last,124.0,475.0,317.0,3.0,259.0,404.0,647.0,1913.0


## 2. Candidatos a sucesión de código de centro

Un cambio administrativo de código se ve como "muere una serie y nace otra", y ninguna de las dos
tendría historia suficiente. Se buscan pares con una regla explícita:

- **A** es un centro que termina antes del último curso del panel.
- **B** es otro centro de la **misma universidad** que empieza el mismo curso en que termina A o el
  siguiente.
- No hay umbral sobre el tamaño: se muestra `size_ratio` (matrícula de B al empezar entre la de A al
  terminar) para que decida una persona.
- `ambiguous` marca los casos con más de un B posible para un A, o más de un A posible para un B.
  Esos no se enlazan automáticamente.

In [4]:
candidates = detect_candidates(panel)
print(f"{len(candidates)} candidatos, {int(candidates['ambiguous'].sum())} ambiguos")
candidates.round({"size_ratio": 2})

22 candidatos, 10 ambiguos


,university,from_center_code,from_center_name,to_center_code,to_center_name,from_last_year,to_first_year,from_last_enrolled,to_first_enrolled,size_ratio,ambiguous
0,Barcelona,08032831,Escuela Universitaria de Enfermería,08072826,Facultad de Medicina y Ciencias de la Salud,2016,2017,1588.0,1557.0,0.98,False
1,Barcelona,08072826,Facultad de Medicina y Ciencias de la Salud,08073481,Facultad de Enfermería,2022,2023,1607.0,1680.0,1.05,False
2,Católica Santa Teresa de Jesús de Ávila,05006521,Facultad de Ciencias y Artes,39014609,Escuela Técnico Profesional en Ciencias de la Salud Clín...,2017,2017,244.0,20.0,0.08,True
3,Católica Santa Teresa de Jesús de Ávila,05006521,Facultad de Ciencias y Artes,05006582,Facultad de Ciencias de la Salud,2017,2018,244.0,284.0,1.16,True
4,Católica de Valencia San Vicente Mártir,46035604,Facultad de Enfermería Nuestra Señora de los Desamparados,46062577,Facultad de Medicina y Ciencias de la Salud,2018,2019,1221.0,1197.0,0.98,False
5,Europea de Madrid,28051852,Facultad de Ciencias Biomédicas y de la Salud. Campus Vi...,28045980,"Facultad de Medicina, Salud y Deportes. Campus Villavici...",2023,2024,575.0,643.0,1.12,False
6,Francisco de Vitoria,28051323,Facultad de Ciencias Biosanitarias,28054282,Facultad de Ciencias de la Salud,2015,2016,489.0,516.0,1.06,False
7,Internacional de la Empresa,28055250,Facultad de Ciencias Sociales Aplicadas y de la Comunica...,28056023,Facultad de Ciencias de la Salud,2024,2025,64.0,190.0,2.97,False
8,La Laguna,38012381,Facultad de Ciencias de la Salud,38012605,Facultad de Enfermería,2024,2025,545.0,537.0,0.99,False
9,La Rioja,26002679,Escuela Universitaria de Enfermería,26003775,Facultad de Ciencias Jurídicas y Sociales,2020,2021,310.0,312.0,1.01,False


## 3. Enlaces confirmados

Las decisiones están en `data/reference/center_links.csv` (versionado en git, con una nota por fila).
`decision` es `sí`, `no` o `dudoso`; **solo `sí` se aplica**. Los `no` y `dudoso` también se guardan
para no volver a preguntar.

Los enlaces son un **juicio**: se decidieron mirando fechas, nombres y tamaños. Un enlace falso une
dos series distintas, por eso cada uno lleva su nota y se puede revisar en un diff de git.

In [5]:
links = load_links(LINKS_FILE)
display(links["decision"].value_counts().rename("filas").to_frame().T)
links

decision,sí,no,dudoso
filas,14,7,1


,university,from_center_code,to_center_code,decision,note
0,Barcelona,08032831,08072826,sí,Cambio de código 2016-17; tamaño 1588 a 1557
1,Barcelona,08072826,08073481,sí,Cambio de código 2022-23 (pasa a Facultad de Enfermería)...
2,Católica Santa Teresa de Jesús de Ávila,05006521,05006582,sí,Cambio de código 2017-18; tamaño 244 a 284
3,Católica Santa Teresa de Jesús de Ávila,05006521,39014609,no,Clínica Mompía (Cantabria) con 20 matriculados: no es su...
4,Católica de Valencia San Vicente Mártir,46035604,46062577,sí,Cambio de código 2018-19; tamaño 1221 a 1197
5,Europea de Madrid,28051852,28045980,sí,Cambio de nombre y código 2023-24 (mismo campus); tamaño...
6,Francisco de Vitoria,28051323,28054282,sí,Cambio de código 2015-16; tamaño 489 a 516
7,Internacional de la Empresa,28055250,28056023,dudoso,El tamaño se triplica (64 a 190): puede ser un centro nu...
8,La Laguna,38012381,38012605,sí,Cambio de código 2024-25 (a Facultad de Enfermería); tam...
9,La Rioja,26002679,26003775,sí,Cambio de código 2020-21; tamaño 310 a 312


## 4. Panel de series

`apply_links` asigna `series_id = "{universidad}|{código final}"`, donde el código final es el más
reciente de la cadena (se admiten cadenas A→B→C). Si dos centros de una serie coinciden en un curso
se suma la matrícula (`min_count=1`), el % de mujeres se pondera por matrícula y el nombre es el del
centro más reciente. Se rechazan los enlaces a centros inexistentes, los ciclos y las bifurcaciones.

Invariante: la matrícula total por curso no cambia al enlazar.

In [6]:
series = apply_links(panel, links)
print(f"{series['series_id'].nunique()} series (antes {panel[['university', 'center_code']].drop_duplicates().shape[0]} centros)")
before = panel.groupby("year_start")["enrolled"].sum()
after = series.groupby("year_start")["enrolled"].sum()
print("Matrícula total por curso idéntica antes y después:", before.equals(after))
series[series["n_source_codes"] > 1][["series_id", "center_name", "n_source_codes"]].drop_duplicates("series_id")

130 series (antes 144 centros)
Matrícula total por curso idéntica antes y después: True


,series_id,center_name,n_source_codes
231,Barcelona|08073481,Facultad de Enfermería,3
396,Católica Santa Teresa de Jesús de Ávila|05006582,Facultad de Ciencias de la Salud,2
418,Católica de Valencia San Vicente Mártir|46062577,Facultad de Medicina y Ciencias de la Salud,2
528,Europea de Madrid|28045980,"Facultad de Medicina, Salud y Deportes. Campus Villavici...",2
616,Francisco de Vitoria|28054282,Facultad de Ciencias de la Salud,2
770,La Laguna|38012605,Facultad de Enfermería,2
781,La Rioja|26004056,Facultad de Ciencias de la Salud,3
1001,País Vasco/Euskal Herriko Unibertsitatea|20009664,Facultad de Medicina y Enfermería. Sección Guipuzcoa,2
1012,País Vasco/Euskal Herriko Unibertsitatea|48012841,Facultad de Medicina y Enfermería,2
1034,Pompeu Fabra|08073491,Centro Universitario Tecnocampus,2


## 5. Calidad de las series

`series_quality` da una fila por serie con: primer y último curso con dato, nº de cursos con dato,
si llega al último curso (`active_last`), tamaño en el último curso, mayor salto interanual
(`max_jump_ratio`, siempre ≥ 1: el cociente en la dirección que más se aleja de 1), nº de saltos por
encima de `JUMP_RATIO` y si tiene centros enlazados.

**Elegible** = llega al último curso **y** tiene al menos `MIN_SEASONS` cursos con dato. No se
compara a través de un curso sin dato, para no fabricar saltos falsos.

In [7]:
quality = series_quality(series)
print("Series:", len(quality), "| activas:", int(quality["active_last"].sum()), "| elegibles:", int(quality["eligible"].sum()))
print("Elegibles sin enlazar:", int(quality_unlinked["eligible"].sum()), "-> con enlaces:", int(quality["eligible"].sum()))
quality["n_seasons"].value_counts().sort_index().rename("series").to_frame().T

Series: 130 | activas: 124 | elegibles: 113
Elegibles sin enlazar: 109 -> con enlaces: 113


n_seasons,1,2,3,4,5,7,8,9,11
series,7,3,4,3,1,2,1,2,107


### Distribución de saltos

Los saltos **no se corrigen**: son crecimiento real de centros que abren grupos o cambios de
adscripción, y borrarlos sería inventar datos. Solo se marcan. El umbral `JUMP_RATIO` se eligió
mirando esta distribución.

In [8]:
display(quality["max_jump_ratio"].describe(percentiles=[0.5, 0.75, 0.9, 0.95]).round(2).to_frame().T)
sensitivity = pd.DataFrame(
    {
        "series_con_algun_salto": [
            int((series_quality(series, jump_ratio=t)["n_jumps"] > 0).sum())
            for t in (1.25, 1.5, 2.0, 3.0)
        ]
    },
    index=pd.Index([1.25, 1.5, 2.0, 3.0], name="jump_ratio"),
)
display(sensitivity.T)
quality.sort_values("max_jump_ratio", ascending=False).head(10)[
    ["series_id", "center_name", "n_seasons", "size_last", "max_jump_ratio", "n_jumps", "eligible"]
].round({"max_jump_ratio": 2})

,count,mean,std,min,50%,75%,90%,95%,max
max_jump_ratio,123.0,1.88,4.49,1.03,1.1,1.27,2.06,2.29,47.0


jump_ratio,1.25,1.50,2.00,3.00
series_con_algun_salto,33,22,14,6


,series_id,center_name,n_seasons,size_last,max_jump_ratio,n_jumps,eligible
9,Antonio de Nebrija|28051891,Centro Universitario de Ciencias de la Salud San Rafael,11,3.0,47.00,2,True
117,València (Estudi General)|12003274,Escuela Universitaria de Enfermería Nuestra Señora del S...,7,2.0,20.00,3,False
16,Autónoma de Barcelona|08044363,Escuela Universitaria de Ciencias de la Salud,5,4.0,6.50,4,False
49,Europea de Madrid|46035631,Centro de Educación Superior Valencia,3,1.0,5.67,2,False
10,Antonio de Nebrija|28055225,Facultad de Ciencias de la Vida y de la Naturaleza,4,395.0,3.50,2,True
84,Málaga|29009417,Centro de Enfermería Diputación Provincial,2,10.0,3.20,1,False
37,Católica Santa Teresa de Jesús de Ávila|39014609,Escuela Técnico Profesional en Ciencias de la Salud Clín...,9,90.0,2.30,1,True
119,València (Estudi General)|46014601,Escuela Universitaria de Enfermería La Fe,11,55.0,2.18,1,True
50,Europea de Valencia|46061731,Facultad de Ciencias de la Salud,11,615.0,2.16,2,True
35,Católica San Antonio|30014108,Centro Universitario UCAM. Cartagena,11,633.0,2.09,1,True


## 6. Series no elegibles

Quedan fuera del conjunto que se podrá prever. El motivo se lee en las dos columnas: `active_last`
falso (la serie termina antes del último curso) o `n_seasons` por debajo de `MIN_SEASONS` (centro
nuevo con poca historia). Las series cortas no se podrán prever con rigor.

In [9]:
not_eligible = quality[~quality["eligible"]].sort_values(["active_last", "n_seasons"])
print(len(not_eligible), "series no elegibles")
not_eligible[["series_id", "center_name", "first_year", "last_year", "n_seasons", "active_last", "size_last"]]

17 series no elegibles


,series_id,center_name,first_year,last_year,n_seasons,active_last,size_last
65,Internacional de la Empresa|28055250,Facultad de Ciencias Sociales Aplicadas y de la Comunica...,2024,2024,1,False,64.0
84,Málaga|29009417,Centro de Enfermería Diputación Provincial,2015,2016,2,False,10.0
49,Europea de Madrid|46035631,Centro de Educación Superior Valencia,2015,2017,3,False,1.0
16,Autónoma de Barcelona|08044363,Escuela Universitaria de Ciencias de la Salud,2015,2019,5,False,4.0
117,València (Estudi General)|12003274,Escuela Universitaria de Enfermería Nuestra Señora del S...,2015,2021,7,False,2.0
82,Murcia|30013116,Escuela Universitaria de Enfermería,2015,2023,9,False,225.0
2,Abat Oliba CEU|08073223,Facultad de Psicología,2025,2025,1,True,80.0
6,Alfonso X el Sabio Mare Nostrum|29016409,Facultad de Salud y Deporte,2025,2025,1,True,61.0
23,CEU Fernando III|41016334,Facultad de Ciencias de la Salud y la Vida,2025,2025,1,True,77.0
66,Internacional de la Empresa|28056023,Facultad de Ciencias de la Salud,2025,2025,1,True,190.0


## 7. Comprobación contra el pipeline

El pipeline (`build_series`) escribe `enfermeria_series.csv`. Debe ser idéntico al panel de series de
este notebook, y la tabla de calidad también.

In [10]:
candidates_p, series_p, quality_p = build_series(panel, LINKS_FILE, out_dir)
from_disk = pd.read_csv(out_dir / "enfermeria_series.csv", dtype={"center_code_final": str})
pd.testing.assert_frame_equal(series.reset_index(drop=True), from_disk, check_dtype=False)
pd.testing.assert_frame_equal(series, series_p)
pd.testing.assert_frame_equal(quality, quality_p)
pd.testing.assert_frame_equal(candidates, candidates_p)
print("El notebook y el pipeline producen exactamente lo mismo.")

El notebook y el pipeline producen exactamente lo mismo.
